# Maintainer's Copilot: pandas Issue Classifier

This notebook trains the Week 7 deep learning classifier on GitHub issues from [`pandas-dev/pandas`](https://github.com/pandas-dev/pandas/issues).

Assignment labels:

- `bug` from GitHub label `bug`
- `feature` from GitHub label `enhancement`
- `docs` from GitHub label `Docs`
- `question` from GitHub label `Usage Question`

## Hand-curated golden set

The repo includes `data/golden/classification_golden_dataset.json` (hand-labeled issues, separate from the time-based test split). This notebook:

1. Loads that golden set **before** building train/val/test splits
2. **Excludes** golden issue IDs from training data (no leakage)
3. Reports metrics on **val**, **time-based test**, and **golden** separately

In Colab, either clone your repo or upload `classification_golden_dataset.json` to `/content/classification_golden_dataset.json`.

After training, download `classifier_artifact.zip`, unzip it locally, and place the files inside your project at `artifacts/classifier/`.

## 1. Runtime setup

In Colab, choose **Runtime -> Change runtime type -> T4 GPU** before running the notebook.

In [1]:
!pip -q install transformers==4.44.2 datasets==2.21.0 evaluate==0.4.2 scikit-learn==1.5.1 requests==2.32.3 accelerate==0.34.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 56.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requ

In [2]:
import json
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import requests
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')

GPU: Tesla T4


## 2. Configuration

A GitHub token is optional but recommended if you fetch many issues. In Colab, you can set it from the left sidebar **Secrets** as `GITHUB_TOKEN`.

In [3]:
OWNER = 'pandas-dev'
REPO = 'pandas'
ISSUE_LIMIT = 2500
BASE_MODEL = 'distilbert-base-uncased'
OUTPUT_DIR = Path('/content/classifier_artifact')
DATA_DIR = Path('/content/pandas_classifier_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['bug', 'feature', 'docs', 'question']
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
GITHUB_TO_ASSIGNMENT_LABEL = {
    'bug': 'bug',
    'enhancement': 'feature',
    'docs': 'docs',
    'usage question': 'question',
}

TRAIN_ARGS = {
    'epochs': 3,
    'batch_size': 16,
    'max_length': 256,
    'learning_rate': 2e-5,
}

# Hand-curated golden set (committed in repo). Checked in order for Colab vs local runs.
GOLDEN_CANDIDATE_PATHS = [
    Path('data/golden/classification_golden_dataset.json'),
    Path('../data/golden/classification_golden_dataset.json'),
    Path('/content/maintainers-copilot/data/golden/classification_golden_dataset.json'),
    Path('/content/classification_golden_dataset.json'),
]

## 3. Load hand-curated golden set

The golden set lives in the repo at `data/golden/classification_golden_dataset.json`. Each row already has a human-reviewed `label` and must **never** appear in train/val/test.

In [5]:
def resolve_golden_path() -> Path:
    for path in GOLDEN_CANDIDATE_PATHS:
        if path.exists():
            return path
    raise FileNotFoundError(
        'Golden set not found. Clone the repo in Colab (recommended) or upload '
        'classification_golden_dataset.json to /content/classification_golden_dataset.json'
    )


def normalize_golden_row(row: dict) -> dict:
    label = row['label']
    if label not in LABELS:
        raise ValueError(f"Golden row {row.get('number')} has invalid label: {label}")
    text = (row.get('text') or f"{row.get('title', '')}\n\n{row.get('body', '')}").strip()
    return {
        'id': row['id'],
        'number': row['number'],
        'title': row.get('title', ''),
        'body': row.get('body', ''),
        'text': text,
        'label': label,
        'source_repo': row.get('source_repo', f'{OWNER}/{REPO}'),
        'source_labels': row.get('source_labels', []),
        'html_url': row.get('html_url'),
        'created_at': row.get('created_at'),
        'closed_at': row.get('closed_at'),
        'sort_date': row.get('closed_at') or row.get('created_at') or '',
        'is_golden': True,
    }


golden_path = resolve_golden_path()
golden_rows = [normalize_golden_row(row) for row in json.loads(golden_path.read_text(encoding='utf-8'))]
golden_issue_ids = {row['id'] for row in golden_rows}
golden_issue_numbers = {row['number'] for row in golden_rows}

print(f'Loaded golden set from {golden_path}')
print(f'Golden examples: {len(golden_rows)}')
print('Golden label distribution:', dict(Counter(row['label'] for row in golden_rows)))

Loaded golden set from /content/classification_golden_dataset.json
Golden examples: 100
Golden label distribution: {'feature': 24, 'bug': 26, 'question': 26, 'docs': 24}


## 4. Fetch closed pandas issues

Pull requests are skipped because GitHub returns PRs from the Issues API too. Issues that appear in the golden set are skipped during dataset construction.

In [7]:
def fetch_closed_issues(owner, repo, limit):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.getenv("GITHUB_TOKEN")

    headers = {
        "Accept": "application/vnd.github+json"
    }

    if token:
        headers["Authorization"] = f"Bearer {token}"

    rows = []
    page = 1

    while len(rows) < limit:
        response = requests.get(
            f"https://api.github.com/repos/{owner}/{repo}/issues",
            headers=headers,
            params={
                "state": "closed",
                "per_page": 100,
                "page": page,
                "sort": "updated",
                "direction": "desc",
                "since": "2020-01-01T00:00:00Z",
            },
            timeout=30,
        )

        if response.status_code == 422:
            print(f"Stopped at page {page} because GitHub rejected further pagination.")
            break

        response.raise_for_status()
        items = response.json()

        if not items:
            break

        for issue in items:
            if "pull_request" in issue:
                continue

            rows.append({
                "id": issue["id"],
                "number": issue["number"],
                "title": issue.get("title") or "",
                "body": issue.get("body") or "",
                "labels": [label.get("name", "") for label in issue.get("labels", [])],
                "state": issue.get("state"),
                "created_at": issue.get("created_at"),
                "closed_at": issue.get("closed_at"),
                "html_url": issue.get("html_url"),
                "comments": issue.get("comments", 0),
            })

            if len(rows) >= limit:
                break

        page += 1

    return rows


raw_issues = fetch_closed_issues(OWNER, REPO, ISSUE_LIMIT)
print(f"Fetched {len(raw_issues)} closed non-PR issues")

(DATA_DIR / "issues.jsonl").write_text(
    "\n".join(json.dumps(row) for row in raw_issues) + "\n",
    encoding="utf-8"
)

Fetched 2500 closed non-PR issues


7680551

## 5. Build time-based dataset (golden held out)

Fetched issues whose `id` or `number` is in the golden set are excluded before the 70/15/15 time split. If an issue has multiple mapped labels, the notebook keeps the first label using assignment label order: bug, feature, docs, question.

In [8]:
def map_issue_labels(github_labels):
    mapped = set()
    for label in github_labels:
        normalized = label.lower().strip()
        if normalized in GITHUB_TO_ASSIGNMENT_LABEL:
            mapped.add(GITHUB_TO_ASSIGNMENT_LABEL[normalized])
    return [label for label in LABELS if label in mapped]


def write_jsonl(path, rows):
    path.write_text('\n'.join(json.dumps(row, ensure_ascii=False) for row in rows) + '\n', encoding='utf-8')


def assert_no_golden_leak(rows: list[dict], split_name: str) -> None:
    leaked_ids = [row['id'] for row in rows if row['id'] in golden_issue_ids]
    leaked_numbers = [row['number'] for row in rows if row['number'] in golden_issue_numbers]
    if leaked_ids or leaked_numbers:
        raise RuntimeError(
            f'{split_name} contains golden issues (ids={leaked_ids[:5]}, numbers={leaked_numbers[:5]}). '
            'Golden rows must stay out of train/val/test.'
        )


dataset = []
warnings = Counter()
for issue in raw_issues:
    if issue['id'] in golden_issue_ids or issue['number'] in golden_issue_numbers:
        warnings['excluded_golden'] += 1
        continue
    mapped = map_issue_labels(issue['labels'])
    if not mapped:
        warnings['no_mapped_label'] += 1
        continue
    if len(mapped) > 1:
        warnings['multiple_mapped_labels'] += 1
    label = mapped[0]
    text = f"{issue['title']}\n\n{issue['body']}".strip()
    dataset.append({
        **issue,
        'label': label,
        'text': text,
        'sort_date': issue.get('closed_at') or issue.get('created_at') or '',
        'source_repo': f'{OWNER}/{REPO}',
        'source_labels': issue['labels'],
        'is_golden': False,
    })

dataset.sort(key=lambda row: row['sort_date'])
train_end = int(len(dataset) * 0.70)
val_end = int(len(dataset) * 0.85)
train_rows = dataset[:train_end]
val_rows = dataset[train_end:val_end]
test_rows = dataset[val_end:]

for split_name, rows in [('train', train_rows), ('val', val_rows), ('test', test_rows)]:
    assert_no_golden_leak(rows, split_name)

write_jsonl(DATA_DIR / 'classification_dataset.jsonl', dataset)
write_jsonl(DATA_DIR / 'train.jsonl', train_rows)
write_jsonl(DATA_DIR / 'val.jsonl', val_rows)
write_jsonl(DATA_DIR / 'test.jsonl', test_rows)
write_jsonl(DATA_DIR / 'classification_golden.jsonl', golden_rows)
(DATA_DIR / 'classification_golden_dataset.json').write_text(
    json.dumps(golden_rows, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

for name, rows in [('all', dataset), ('train', train_rows), ('val', val_rows), ('test', test_rows), ('golden', golden_rows)]:
    counts = Counter(row['label'] for row in rows)
    print(name, len(rows), dict(counts))
print('warnings', dict(warnings))
print('golden held out from training:', warnings['excluded_golden'])

all 1886 {'bug': 1075, 'feature': 423, 'question': 28, 'docs': 360}
train 1320 {'bug': 748, 'feature': 320, 'question': 20, 'docs': 232}
val 283 {'feature': 43, 'docs': 55, 'bug': 183, 'question': 2}
test 283 {'bug': 144, 'docs': 73, 'feature': 60, 'question': 6}
golden 100 {'feature': 24, 'bug': 26, 'question': 26, 'docs': 24}
warnings {'no_mapped_label': 529, 'multiple_mapped_labels': 15, 'excluded_golden': 85}
golden held out from training: 85


## 6. Tokenize and train DistilBERT

In [9]:
class IssueDataset(torch.utils.data.Dataset):
    def __init__(self, rows, tokenizer, max_length):
        self.rows = rows
        self.encodings = tokenizer(
            [row['text'] for row in rows],
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        self.labels = [LABEL_TO_ID[row['label']] for row in rows]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

train_ds = IssueDataset(train_rows, tokenizer, TRAIN_ARGS['max_length'])
val_ds = IssueDataset(val_rows, tokenizer, TRAIN_ARGS['max_length'])
test_ds = IssueDataset(test_rows, tokenizer, TRAIN_ARGS['max_length'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
def metric_bundle(predictions, labels):
    pred_ids = np.argmax(predictions, axis=1)
    per_class = f1_score(labels, pred_ids, average=None, labels=list(range(len(LABELS))), zero_division=0)
    return {
        'accuracy': float(accuracy_score(labels, pred_ids)),
        'macro_f1': float(f1_score(labels, pred_ids, average='macro', zero_division=0)),
        'per_class_f1': {label: float(score) for label, score in zip(LABELS, per_class)},
        'confusion_matrix': confusion_matrix(labels, pred_ids, labels=list(range(len(LABELS)))).tolist(),
        'labels': LABELS,
    }

def compute_metrics(eval_pred):
    metrics = metric_bundle(eval_pred.predictions, eval_pred.label_ids)
    return {
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        **{f"f1_{label}": score for label, score in metrics['per_class_f1'].items()},
    }

training_args = TrainingArguments(
    output_dir='/content/classifier_checkpoints',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=TRAIN_ARGS['learning_rate'],
    per_device_train_batch_size=TRAIN_ARGS['batch_size'],
    per_device_eval_batch_size=TRAIN_ARGS['batch_size'],
    num_train_epochs=TRAIN_ARGS['epochs'],
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,F1 Bug,F1 Feature,F1 Docs,F1 Question
1,0.782100,0.292829,0.915194,0.667125,0.950549,0.833333,0.884615,0.000000
2,0.311800,0.246746,0.939929,0.691496,0.965147,0.898876,0.901961,0.000000
3,0.341600,0.249758,0.915194,0.670162,0.945355,0.833333,0.901961,0.000000


TrainOutput(global_step=249, training_loss=0.4302397731796326, metrics={'train_runtime': 112.3096, 'train_samples_per_second': 35.26, 'train_steps_per_second': 2.217, 'total_flos': 262294804316160.0, 'train_loss': 0.4302397731796326, 'epoch': 3.0})

## 7. Evaluate and export artifacts

Metrics are reported for validation, the time-based test split, and the **held-out golden set** (`classification_golden_dataset.json`).

In [11]:
golden_ds = IssueDataset(golden_rows, tokenizer, TRAIN_ARGS['max_length'])

val_pred = trainer.predict(val_ds)
test_pred = trainer.predict(test_ds)
golden_pred = trainer.predict(golden_ds)
val_metrics = metric_bundle(val_pred.predictions, val_pred.label_ids)
test_metrics = metric_bundle(test_pred.predictions, test_pred.label_ids)
golden_metrics = metric_bundle(golden_pred.predictions, golden_pred.label_ids)

print('Validation metrics (time-based val split)')
print(json.dumps(val_metrics, indent=2))
print('Test metrics (time-based test split)')
print(json.dumps(test_metrics, indent=2))
print('Golden metrics (hand-curated held-out set)')
print(json.dumps(golden_metrics, indent=2))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR / 'label_mapping.json').write_text(json.dumps({
    'label_to_id': LABEL_TO_ID,
    'id_to_label': {str(key): value for key, value in ID_TO_LABEL.items()},
}, indent=2) + '\n')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(val_metrics, indent=2) + '\n')
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2) + '\n')
(OUTPUT_DIR / 'golden_metrics.json').write_text(json.dumps(golden_metrics, indent=2) + '\n')
(OUTPUT_DIR / 'training_config.json').write_text(json.dumps({
    'dataset_source': f'{OWNER}/{REPO}',
    'golden_set_path': str(golden_path),
    'golden_set_size': len(golden_rows),
    'golden_excluded_from_training': True,
    'github_label_mapping': GITHUB_TO_ASSIGNMENT_LABEL,
    'base_model': BASE_MODEL,
    **TRAIN_ARGS,
    'train_size': len(train_rows),
    'val_size': len(val_rows),
    'test_size': len(test_rows),
    'labels': LABELS,
}, indent=2) + '\n')

zip_path = Path('/content/classifier_artifact.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.rglob('*'):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(OUTPUT_DIR))
print(f'Wrote {zip_path}')
print('Primary CI eval uses time-based test split (scripts/evaluate_classifier.py).')
print('Golden metrics are in golden_metrics.json for assignment reporting.')

Validation metrics (time-based val split)
{
  "accuracy": 0.9399293286219081,
  "macro_f1": 0.6914961604728044,
  "per_class_f1": {
    "bug": 0.9651474530831099,
    "feature": 0.898876404494382,
    "docs": 0.9019607843137255,
    "question": 0.0
  },
  "confusion_matrix": [
    [
      180,
      3,
      0,
      0
    ],
    [
      2,
      40,
      1,
      0
    ],
    [
      6,
      3,
      46,
      0
    ],
    [
      2,
      0,
      0,
      0
    ]
  ],
  "labels": [
    "bug",
    "feature",
    "docs",
    "question"
  ]
}
Test metrics (time-based test split)
{
  "accuracy": 0.8798586572438163,
  "macro_f1": 0.6607590670053964,
  "per_class_f1": {
    "bug": 0.9073482428115016,
    "feature": 0.907563025210084,
    "docs": 0.828125,
    "question": 0.0
  },
  "confusion_matrix": [
    [
      142,
      2,
      0,
      0
    ],
    [
      5,
      54,
      1,
      0
    ],
    [
      17,
      3,
      53,
      0
    ],
    [
      5,
      0,
      1,
    

## 8. Download and copy back to the project

Download `/content/classifier_artifact.zip` from Colab.

On your laptop, unzip it so the files are directly inside:

```text
artifacts/classifier/
  config.json
  model.safetensors or pytorch_model.bin
  tokenizer_config.json
  tokenizer.json
  vocab.txt
  special_tokens_map.json
  label_mapping.json
  metrics.json
  test_metrics.json
  golden_metrics.json
  training_config.json
```

Optional: copy refreshed splits from Colab `DATA_DIR` into the repo:

```text
data/processed/train.jsonl
data/processed/val.jsonl
data/processed/test.jsonl
```

The golden set stays at `data/golden/classification_golden_dataset.json` (do not merge into train).

Then run locally:

```bash
python3 scripts/evaluate_classifier.py
uvicorn model_server.main:app --host 0.0.0.0 --port 8001
```

The model server `/classify` endpoint will load from `artifacts/classifier/`.